# Headline PCE (month-over-month): Five Regression Models, In-Sample vs Out-of-Sample

**Goal:** find the best regression model for the monthly % change in headline PCE.

**Data:** `fred_monthly_levels.csv` (same folder as this notebook, or in a `data/` subfolder). Nothing is downloaded.

**Train/test split (time-ordered, no shuffling):**

| | Months | Period |
|---|---|---|
| Train | 191 (82%) | Feb 2007 to Dec 2022 (includes all of COVID) |
| Test | 42 (18%) | Jan 2023 to Aug 2026 (no COVID months) |

All tuning happens **inside the training months only**. The test months are used once, at the end, to score the models.

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, HuberRegressor
from sklearn.decomposition import PCA
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

LEVELS_PATH = next((p for p in [Path("fred_monthly_levels.csv"), Path("data/fred_monthly_levels.csv")] if p.exists()), None)
assert LEVELS_PATH is not None, "Put fred_monthly_levels.csv next to this notebook (or in a data/ folder)."

SAMPLE_START = "2007-02-01"                 # start of the sample
TRAIN_END    = pd.Timestamp("2022-12-01")   # last training month; everything after is the test set
COVID        = "covid_2020_2022"

## 1. Load the data

In [2]:
levels = pd.read_csv(LEVELS_PATH, index_col="month", parse_dates=True).asfreq("MS")
print("Levels file:", levels.shape, "|", levels.index.min().date(), "to", levels.index.max().date())

Levels file: (442, 48) | 1990-01-01 to 2026-10-01


## 2. Transformations (only what the models need)
* Every series is converted from a **level** to a **monthly % change**.
* **Target:** monthly % change in headline PCE.
* **Predictors** (naming: `_L0` = same month, `_L1` = previous month, `_L2`/`_L3` = two/three months back):
  * past PCE and core PCE changes (lags 2 and 3, because PCE is released with a delay)
  * gasoline and oil: last month's change (L1) and same-month change (L0)
  * same-month natural gas, dollar index and Nasdaq changes (L0)
  * CPI, PPI, import-price and earnings changes, lagged one month (L1)
  * a COVID dummy (1 for Mar 2020 to Dec 2022)
* Months with a missing predictor are dropped (Nov and Dec 2025, because October 2025 CPI was never published).

**Timing note:** the same-month (`_L0`) variables are only known once a month has ended, so these models assume a forecast made after the month closes.

In [3]:
chg = 100 * (levels / levels.shift(1) - 1)                 # monthly % change of every series
target = chg["pce_headline"].rename("y_pce_mom")
core   = chg["pce_core"]

MARKET = ["retail_gasoline_weekly", "wti_oil", "henry_hub_gas", "usd_broad_index", "nasdaq_composite"]
release_cols = [c for c in levels.columns
                if not c.startswith(("pce_", "nominal_", "real_")) and c != "sp500" and c not in MARKET]

X = pd.DataFrame(index=levels.index)
X["pce_L2"],      X["pce_L3"]      = target.shift(2), target.shift(3)
X["core_pce_L2"], X["core_pce_L3"] = core.shift(2),   core.shift(3)
for c in MARKET:
    X[c + "_L0"] = chg[c]                                  # same-month change
for c in ["retail_gasoline_weekly", "wti_oil"]:
    X[c + "_L1"] = chg[c].shift(1)                         # last month's change
for c in release_cols:
    X[c + "_L1"] = chg[c].shift(1)                         # CPI / PPI / import prices / earnings
X[COVID] = ((X.index >= "2020-03-01") & (X.index <= "2022-12-01")).astype(float)

# the two predictor lists used below
BASE    = ["pce_L2", "pce_L3", "core_pce_L2", "core_pce_L3",
           "retail_gasoline_weekly_L1", "wti_oil_L1"] + [c + "_L0" for c in MARKET]
CPI_KEY = ["cpi_headline_L1", "cpi_core_L1", "cpi_core_goods_L1", "cpi_new_vehicles_L1",
           "cpi_used_cars_L1", "cpi_oer_L1", "cpi_airline_fares_L1", "cpi_services_less_energy_L1"]
S3 = BASE + CPI_KEY + [COVID]                               # 20 variables
S4 = BASE + [c + "_L1" for c in release_cols] + [COVID]     # 41 variables (adds every CPI/PPI/import/earnings series)

window = pd.concat([target, X], axis=1).loc[SAMPLE_START:target.last_valid_index()]
data = window.dropna(subset=["y_pce_mom"] + S4)
print(f"Usable months: {len(data)} ({data.index.min():%Y-%m} to {data.index.max():%Y-%m})")
print("Dropped for missing predictors:", [d.strftime("%Y-%m") for d in window.index.difference(data.index)])
print(f"S3 = {len(S3)} variables | S4 = {len(S4)} variables")

Usable months: 233 (2007-02 to 2026-08)
Dropped for missing predictors: ['2025-11', '2025-12']
S3 = 20 variables | S4 = 41 variables


## 3. Train / test split

In [4]:
train = data[data.index <= TRAIN_END]
test  = data[data.index >  TRAIN_END]
y_tr, y_te = train["y_pce_mom"], test["y_pce_mom"]
train_mean = y_tr.mean()          # the "predict the training average" benchmark

print(f"Train: {len(train)} months ({len(train)/len(data):.0%}), {train.index.min():%b %Y} to {train.index.max():%b %Y} | COVID months: {int(train[COVID].sum())}")
print(f"Test : {len(test)} months ({len(test)/len(data):.0%}), {test.index.min():%b %Y} to {test.index.max():%b %Y} | COVID months: {int(test[COVID].sum())}")

Train: 191 months (82%), Feb 2007 to Dec 2022 | COVID months: 34
Test : 42 months (18%), Jan 2023 to Aug 2026 | COVID months: 0


## 4. The five models
These are the five best results from the earlier full comparison (10 model types x 5 feature sets):

| # | Model | Predictors | How it works |
|---|---|---|---|
| 1 | **Forward stepwise + OLS** | S3 (20) | Adds the most useful variable one at a time, then fits ordinary least squares on the chosen few |
| 2 | **Huber** | S3 (20) | Regression that gives extreme months less influence (robust to outliers) |
| 3 | **Huber** | S4 (41) | Same model with every CPI/PPI/import/earnings series added |
| 4 | **PCR** (principal component regression) | S3 (20) | Compresses the predictors into a few uncorrelated components, then runs OLS on those |
| 5 | **Ridge** | S3 (20) | OLS with a penalty that shrinks coefficients to limit overfitting |

Each model standardizes its variables first. Its tuning setting (variables kept, penalty strength, number of components) is chosen by forward-chaining validation **inside the training months only**.

**Metrics** (computed twice: on the training months = in-sample, and on the test months = out-of-sample):
* **RMSE**: typical size of the miss, in percentage points (bigger misses count more). Lower is better.
* **MAE**: average absolute miss. Lower is better.
* **R²**: share of the error of "always predict the training average" that the model removes. 1 is perfect, 0 is no better than that benchmark, negative is worse.

In [5]:
inner_cv = TimeSeriesSplit(n_splits=5)      # forward-chaining validation used for tuning (training months only)

MODELS = {
    "1. Forward stepwise + OLS": dict(set="S3", cols=S3,
        pipe=make_pipeline(StandardScaler(),
                           SequentialFeatureSelector(LinearRegression(), direction="forward", cv=3),
                           LinearRegression()),
        grid={"sequentialfeatureselector__n_features_to_select": [3, 6, 10]}),
    "2. Huber (S3)": dict(set="S3", cols=S3,
        pipe=make_pipeline(StandardScaler(), HuberRegressor(max_iter=5000)),
        grid={"huberregressor__alpha": np.logspace(-3, 3, 10), "huberregressor__epsilon": [1.35, 1.75]}),
    "3. Huber (S4)": dict(set="S4", cols=S4,
        pipe=make_pipeline(StandardScaler(), HuberRegressor(max_iter=5000)),
        grid={"huberregressor__alpha": np.logspace(-3, 3, 10), "huberregressor__epsilon": [1.35, 1.75]}),
    "4. PCR (S3)": dict(set="S3", cols=S3,
        pipe=make_pipeline(StandardScaler(), PCA(), LinearRegression()),
        grid={"pca__n_components": [1, 2, 3, 5, 8, 12]}),
    "5. Ridge (S3)": dict(set="S3", cols=S3,
        pipe=make_pipeline(StandardScaler(), Ridge()),
        grid={"ridge__alpha": np.logspace(-1, 4, 20)}),
}

def metrics(y_true, y_pred):
    err = y_true - y_pred
    sse_benchmark = ((y_true - train_mean) ** 2).sum()      # error of "predict the training average"
    return {"RMSE": float(np.sqrt((err ** 2).mean())),
            "MAE":  float(err.abs().mean()),
            "R2":   float(1 - (err ** 2).sum() / sse_benchmark)}

def describe_setting(params):
    return ", ".join(f"{k.split('__')[1]} = {v:.4g}" if isinstance(v, (float, np.floating)) else f"{k.split('__')[1]} = {v}"
                     for k, v in params.items())

rows = []
for name, spec in MODELS.items():
    cols = spec["cols"]
    search = GridSearchCV(spec["pipe"], spec["grid"], cv=inner_cv, scoring="neg_mean_squared_error", n_jobs=-1)
    search.fit(train[cols], y_tr)                                   # tuning sees ONLY the training months
    pred_in  = pd.Series(np.ravel(search.predict(train[cols])), index=train.index)
    pred_out = pd.Series(np.ravel(search.predict(test[cols])),  index=test.index)
    m_in, m_out = metrics(y_tr, pred_in), metrics(y_te, pred_out)

    n_used = len(cols)
    steps = search.best_estimator_.named_steps
    if "sequentialfeatureselector" in steps:
        keep = np.array(cols)[steps["sequentialfeatureselector"].get_support()]
        n_used = len(keep)
        print(f"{name}: kept variables -> {', '.join(keep)}")
    print(f"{name}: tuned setting -> {describe_setting(search.best_params_)}")

    rows.append({"model": name, "predictors": spec["set"], "n_variables": n_used,
                 "setting": describe_setting(search.best_params_),
                 "in_RMSE": m_in["RMSE"], "in_MAE": m_in["MAE"], "in_R2": m_in["R2"],
                 "out_RMSE": m_out["RMSE"], "out_MAE": m_out["MAE"], "out_R2": m_out["R2"]})

res = pd.DataFrame(rows).set_index("model").sort_values("out_RMSE")
print("\nDone. All five models fitted on the training months and scored on both samples.")

1. Forward stepwise + OLS: kept variables -> pce_L3, retail_gasoline_weekly_L0, usd_broad_index_L0, cpi_core_L1, cpi_new_vehicles_L1, covid_2020_2022
1. Forward stepwise + OLS: tuned setting -> n_features_to_select = 6
2. Huber (S3): tuned setting -> alpha = 215.4, epsilon = 1.35


/Users/aca/Downloads/Mcdonalds EDA/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/aca/Downloads/Mcdonalds EDA/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/aca/Downloads/Mcdonalds EDA/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/aca/Downloads/Mcdonalds EDA/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/aca/Downloads/Mcdonalds EDA/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/aca/Downloads/Mcdonalds EDA/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/aca

3. Huber (S4): tuned setting -> alpha = 1000, epsilon = 1.75
4. PCR (S3): tuned setting -> n_components = 12
5. Ridge (S3): tuned setting -> alpha = 42.81

Done. All five models fitted on the training months and scored on both samples.


## 5. Summary of the five models and their results
Ranked by **out-of-sample RMSE** (error on the 42 test months the models never saw).

In [6]:
summary = res[["predictors", "n_variables", "setting", "out_RMSE", "out_MAE", "out_R2"]].copy()
summary.columns = ["Predictors", "Variables used", "Tuned setting", "Test RMSE", "Test MAE", "Test R²"]
summary.insert(0, "Rank", range(1, len(summary) + 1))

bench_out = float(np.sqrt(((y_te - train_mean) ** 2).mean()))
print(f"Benchmark (always predict the training average, {train_mean:.3f}%): test RMSE = {bench_out:.4f}\n")
summary.round(4)

Benchmark (always predict the training average, 0.172%): test RMSE = 0.1499



,Rank,Predictors,Variables used,Tuned setting,Test RMSE,Test MAE,Test R²
model,,,,,,,
2. Huber (S3),1,S3,20,"alpha = 215.4, epsilon = 1.35",0.0841,0.0671,0.6856
3. Huber (S4),2,S4,41,"alpha = 1000, epsilon = 1.75",0.0856,0.0650,0.6737
5. Ridge (S3),3,S3,20,alpha = 42.81,0.0867,0.0707,0.6657
4. PCR (S3),4,S3,20,n_components = 12,0.1015,0.0799,0.5421
1. Forward stepwise + OLS,5,S3,6,n_features_to_select = 6,0.1058,0.0799,0.5023


## 6. In-sample vs out-of-sample, side by side
* **In-sample** = the models scored on the 191 training months (data they were fitted on).
* **Out-of-sample** = the same fitted models scored on the 42 test months (data they never saw).
* The last column is the **R² gap** (in-sample minus out-of-sample). A larger gap means the in-sample score was flattering the model more.
* The last row is the benchmark: "always predict the training average" (R² = 0 by definition).

**How to read it:** compare **R²** first. The training period includes COVID, which was far more volatile than 2023 to 2026, so raw RMSE and MAE are not directly comparable between the two samples. R² is measured relative to the benchmark within each sample, so it is.

In [7]:
side = pd.DataFrame({
    ("In-sample (train)",     "RMSE"): res["in_RMSE"],
    ("In-sample (train)",     "MAE"):  res["in_MAE"],
    ("In-sample (train)",     "R²"):   res["in_R2"],
    ("Out-of-sample (test)",  "RMSE"): res["out_RMSE"],
    ("Out-of-sample (test)",  "MAE"):  res["out_MAE"],
    ("Out-of-sample (test)",  "R²"):   res["out_R2"],
    ("Gap", "R² in-sample minus out-of-sample"): res["in_R2"] - res["out_R2"],
})

bench_in = {"RMSE": float(np.sqrt(((y_tr - train_mean) ** 2).mean())), "MAE": float((y_tr - train_mean).abs().mean())}
bench_te = {"RMSE": bench_out, "MAE": float((y_te - train_mean).abs().mean())}
side.loc["Benchmark: predict training average"] = [bench_in["RMSE"], bench_in["MAE"], 0.0,
                                                   bench_te["RMSE"], bench_te["MAE"], 0.0, 0.0]
side.round(3)

In-sample (train)                \
                                                 RMSE    MAE     R²   
model                                                                 
2. Huber (S3)                                   0.108  0.076  0.793   
3. Huber (S4)                                   0.119  0.082  0.749   
5. Ridge (S3)                                   0.108  0.080  0.795   
4. PCR (S3)                                     0.116  0.088  0.763   
1. Forward stepwise + OLS                       0.112  0.085  0.777   
Benchmark: predict training average             0.238  0.158  0.000   

                                    Out-of-sample (test)                \
                                                    RMSE    MAE     R²   
model                                                                    
2. Huber (S3)                                      0.084  0.067  0.686   
3. Huber (S4)                                      0.086  0.065  0.674   
5. Ridge (S3)                                      0.087  0.071  0.666   
4. PCR (S3)                                        0.101  0.080  0.542   
1. Forward stepwise + OLS                          0.106  0.080  0.502   
Benchmark: predict training average                0.150  0.118  0.000   

                                                                 Gap  
                                    R² in-sample minus out-of-sample  
model                                                                 
2. Huber (S3)                                                  0.107  
3. Huber (S4)                                                  0.075  
5. Ridge (S3)                                                  0.129  
4. PCR (S3)                                                    0.221  
1. Forward stepwise + OLS                                      0.275  
Benchmark: predict training average                            0.000

## 7. What variables were actually used in each of the final models

Listed below

In [8]:
from IPython.display import display

def refit(spec):
    """Refit one model exactly as in the model cell (tuned on the training months only)."""
    search = GridSearchCV(spec["pipe"], spec["grid"], cv=inner_cv, scoring="neg_mean_squared_error", n_jobs=-1)
    search.fit(train[spec["cols"]], y_tr)
    return search.best_estimator_

def standardized_coefs(pipe, cols):
    """Coefficient on each STANDARDIZED variable = change in predicted monthly PCE (percentage points)
    for a 1 standard-deviation move in that variable. Variables a model does not use are left out."""
    steps, cols = pipe.named_steps, np.array(cols)
    if "sequentialfeatureselector" in steps:                 # stepwise: only the kept variables
        keep = cols[steps["sequentialfeatureselector"].get_support()]
        return pd.Series(steps["linearregression"].coef_, index=keep)
    if "pca" in steps:                                       # PCR: map the component weights back to the variables
        w = steps["pca"].components_.T @ steps["linearregression"].coef_
        return pd.Series(w, index=cols)
    return pd.Series(np.ravel(pipe.steps[-1][1].coef_), index=cols)   # Huber / Ridge

coefs = {}
for name, spec in MODELS.items():
    coefs[name] = standardized_coefs(refit(spec), spec["cols"])
    print(f"{name}: {len(coefs[name])} variables used")

print("\nVariables kept by forward stepwise:")
print(", ".join(coefs["1. Forward stepwise + OLS"].index))

# rows = variables, columns = models, blank = the model does not use that variable
matrix = pd.DataFrame(coefs)
matrix = matrix.loc[matrix.abs().mean(axis=1).sort_values(ascending=False).index]
print("\nStandardized coefficients (sorted by average size across models):")
matrix.round(3).fillna("")

1. Forward stepwise + OLS: 6 variables used
2. Huber (S3): 20 variables used


/Users/aca/Downloads/Mcdonalds EDA/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/aca/Downloads/Mcdonalds EDA/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/aca/Downloads/Mcdonalds EDA/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/aca/Downloads/Mcdonalds EDA/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/aca/Downloads/Mcdonalds EDA/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/aca/Downloads/Mcdonalds EDA/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/aca

3. Huber (S4): 41 variables used
4. PCR (S3): 20 variables used
5. Ridge (S3): 20 variables used

Variables kept by forward stepwise:
pce_L3, retail_gasoline_weekly_L0, usd_broad_index_L0, cpi_core_L1, cpi_new_vehicles_L1, covid_2020_2022

Standardized coefficients (sorted by average size across models):


,1. Forward stepwise + OLS,2. Huber (S3),3. Huber (S4),4. PCR (S3),5. Ridge (S3)
retail_gasoline_weekly_L0,0.161,0.106,0.065,0.079,0.099
wti_oil_L0,,0.029,0.031,0.074,0.035
cpi_headline_L1,,0.046,0.019,0.017,0.038
covid_2020_2022,0.036,0.029,0.014,0.02,0.024
usd_broad_index_L0,-0.028,-0.02,-0.020,-0.014,-0.023
pce_L3,0.042,0.02,0.007,0.011,0.023
cpi_airline_fares_L1,,0.018,0.018,0.027,0.017
cpi_new_vehicles_L1,0.021,0.015,0.013,0.016,0.019
cpi_core_L1,0.046,0.005,0.010,0.013,0.008
wti_oil_L1,,0.008,0.018,0.021,0.013


## 8. Why out-of-sample is the right way to judge a forecasting model

**The short version:** in-sample, the model is graded on the same months it was trained on, so it is being tested on data it has already seen. Our real task is the opposite: predicting a month that does not exist yet. Only out-of-sample scores, which use months the model never saw (here, Jan 2023 to Aug 2026), reproduce that situation, so they give a more accurate picture of how the model will really perform.

| | **In-sample** (train months) | **Out-of-sample** (test months) |
|---|---|---|
| **What is scored** | Months the model was fitted on, which it has already seen | Months held back that the model has never seen |
| **What a good score proves** | The model can describe the past | The model can predict new months, which is what a forecast needs |
| **Effect of random noise** | The model partly fits the noise in these months, which flatters the score | Noise in new months cannot be predicted, so it does not flatter the score |
| **Effect of adding variables** | For ordinary least squares, the in-sample score never gets worse as variables are added, even if they are useless | Useless variables usually make the score worse, which exposes overfitting |
| **Effect of tuning and variable selection** | Settings and variables were chosen to fit these very months | Test months are never used for any choice, so there is no hidden advantage |
| **Match to the real task** | Forecasting November 2026 means predicting a month no model has seen, so this does not match | Matches the task, and in time order (train on the past, predict the future) |
| **Regime changes** | Cannot show whether relationships still hold after the training data ends | Tests months after the training period (post-COVID), where relationships may have shifted |
| **Direction of the error** | Tends to be too optimistic | Honest estimate of real forecasting accuracy |
| **How to read the gap** | A big in-sample advantage over out-of-sample signals overfitting (the model memorized the training months) | A small gap means the model generalizes |